# ***Import necesario***

In [ ]:
import ROOT as rt

# ***Resumen***

Para este ejercicio realizaremos una comparación entre datos experimentales y simulación teórica (proceso Drell-Yan) analizando dos variables críticas: energía faltante en el plano transverso (MET) y masa invariante del sistema di-muón. Utilizaremos histogramas superpuestos y gráficos de ratio (cociente) para evaluar el acuerdo entre teoría y experimento.

# ***Procedimiento***

## Descripción del análisis

Este ejercicio compara datos experimentales con simulaciones del **proceso Drell-Yan** (producción de pares muón-antimuón mediante aniquilación quark-antiquark). Trabajamos con dos archivos ROOT:

- **dy.root**: Simulación Monte Carlo del proceso Drell-Yan
- **data.root**: Datos experimentales reales

## Variables analizadas

### Energía Faltante Transversa (MET)
Calculada como $\text{MET} = \sqrt{\text{MET}_{\text{px}}^2 + \text{MET}_{\text{py}}^2}$, representa la energía no detectada en el plano perpendicular al haz. Útil para detectar neutrinos u otras partículas que no interactúan con el detector.

### Masa Invariante Di-muón
Obtenida sumando los cuadrivectores de los dos muones: $M = \sqrt{(E_1+E_2)^2 - (\vec{p}_1+\vec{p}_2)^2}$. Esta cantidad es invariante bajo transformaciones de Lorentz y permite reconstruir la masa de la partícula madre (en este caso, el bosón Z o fotones virtuales).

## Configuración de histogramas

**Parámetros comunes:**
- **30 bines**: Balance entre resolución estadística y capacidad de observar tendencias
- **Rango MET**: $0-80$ GeV (ajustado tras observar la distribución real de los datos)
- **Rango Masa**: $0-140$ GeV (incluye el pico del bosón Z a $\sim 91$ GeV)
- **Normalización**: Todos los histogramas se escalan dividiéndolos por su integral, permitiendo comparar formas independientemente del número de eventos

## Selecciones aplicadas

- **Eventos con exactamente 2 muones** (`NMuon == 2`): Firma típica del proceso Drell-Yan
- Los muones deben tener cargas opuestas (implícito en el proceso DY)

## Gráficos de Ratio

Los gráficos de ratio (Datos/DY) son fundamentales en física de partículas:
- **Ratio = 1**: Acuerdo perfecto entre datos y simulación
- **Ratio > 1**: Exceso de eventos en datos respecto a la teoría
- **Ratio < 1**: Déficit de eventos en datos respecto a la teoría
- **Barras de error**: Muestran la incertidumbre estadística propagada usando la fórmula $\sigma_{\text{ratio}} = R \times \sqrt{\left(\frac{\sigma_D}{D}\right)^2 + \left(\frac{\sigma_{DY}}{DY}\right)^2}$
- **Línea roja en $y=1$**: Referencia visual para identificar desviaciones significativas

# ***Representación y código***

## Estructura del canvas

El código genera un lienzo dividido en **4 pads (2×2)**:

1. **Pad 1** (superior izquierdo): Comparación MET (DY en azul vs Datos en rojo)
2. **Pad 2** (superior derecho): Ratio MET (Datos/DY) con barras de error
3. **Pad 3** (inferior izquierdo): Comparación Masa Invariante (DY en verde vs Datos en magenta)
4. **Pad 4** (inferior derecho): Ratio Masa Invariante (Datos/DY) con barras de error

## Archivos utilizados

- **Ficheros**: dy.root y data.root
- **Árbol**: "events" (contiene información de cada colisión)
- **Bins**: 30 bins para ambas variables
- **Rangos**: MET $[0-80 \text{ GeV}]$, Masa Invariante $[0-140 \text{ GeV}]$
- **Selecciones**: `NMuon == 2` (eventos con exactamente 2 muones)

# ***Anotaciones***

Las barras de error vienen dadas por la propagación de errores que ya incluye la función de ROOT, luego el proceso del codigo es muy similar a todo lo que hicimos en el ejercicio anterior, algunos problemas que nos hemos encontrado es que importa el orden en el que cargamos los ficheros.

# ***Imagen***

# ***Código***

In [ ]:
# ========================================
# PARTE 1: CARGAR DATOS DE SIMULACIÓN DY
# ========================================
dy = rt.TFile("../Datos/dy.root")
t = dy.Get("events")  # Obtener el árbol de eventos de la simulación Drell-Yan

# Crear canvas dividido en 4 pads (2x2)
canvas = rt.TCanvas("canvas", "Análisis DY vs Datos", 1600, 1000)
canvas.Divide(2,2)  # 2 columnas, 2 filas

# ========================================
# HISTOGRAMA 1: MET para DY (h)
# ========================================
h = rt.TH1F("EFPT0", "Energía Faltante plano trasverso", 30, 0, 80)
h.SetXTitle("MET (GeV)")
h.SetYTitle("cantidad de eventos")
h.SetLineColor(rt.kBlue)  # Color azul para DY
h.SetLineWidth(2)
h.SetStats(0)  # Ocultar cuadro de estadísticas

# ========================================
# HISTOGRAMA 2: Masa invariante para DY (h_masa)
# ========================================
h_masa = rt.TH1F("masa", "Masa invariante di-muon", 30, 0, 140)
h_masa.SetXTitle("M (GeV)")
h_masa.SetYTitle("cantidad de eventos")
h_masa.SetLineColor(rt.kGreen)  # Color verde para DY
h_masa.SetLineWidth(2)
h_masa.SetStats(0)

# Llenar histograma de MET para eventos con 2 muones
t.Draw("sqrt(MET_px*MET_px + MET_py*MET_py)>>EFPT0","NMuon == 2","goff")

# Calcular masa invariante di-muon para eventos DY
for entry in t:
    if entry.NMuon == 2:  # Solo eventos con exactamente 2 muones
        # Crear vectores 4-momento para cada muón
        Muon1 = rt.TLorentzVector()
        Muon1.SetPxPyPzE(entry.Muon_Px[0], entry.Muon_Py[0], entry.Muon_Pz[0], entry.Muon_E[0])
        Muon2 = rt.TLorentzVector()
        Muon2.SetPxPyPzE(entry.Muon_Px[1], entry.Muon_Py[1], entry.Muon_Pz[1], entry.Muon_E[1])
        # Sumar vectores para obtener el sistema di-muon
        diMuon = Muon1 + Muon2
        h_masa.Fill(diMuon.M())  # Llenar con la masa invariante


# ========================================
# PARTE 2: CARGAR DATOS EXPERIMENTALES
# ========================================
data = rt.TFile("../Datos/data.root")
t = data.Get("events")  # Obtener el árbol de datos experimentales

# ========================================
# HISTOGRAMA 3: MET para Datos (h2)
# ========================================
h2 = rt.TH1F("EFPT1", "Energía Faltante plano trasverso", 30, 0, 80)
h2.SetXTitle("MET (GeV)")
h2.SetYTitle("cantidad de eventos")
h2.SetLineColor(rt.kRed)  # Color rojo para datos
h2.SetLineWidth(2)
h2.SetStats(0)
t.Draw("sqrt(MET_px*MET_px + MET_py*MET_py)>>EFPT1","NMuon == 2","goff")

# ========================================
# HISTOGRAMA 4: Masa invariante para Datos (h_masa2)
# ========================================
h_masa2 = rt.TH1F("masa2", "Masa invariante 2 muones", 30, 0, 140)
h_masa2.SetXTitle("M (GeV)")
h_masa2.SetYTitle("cantidad de eventos")
h_masa2.SetLineColor(rt.kMagenta)  # Color magenta para datos
h_masa2.SetLineWidth(2)
h_masa2.SetStats(0)

# Calcular masa invariante di-muon para datos experimentales
for entry in t:
    if entry.NMuon == 2:
        Muon1 = rt.TLorentzVector()
        Muon1.SetPxPyPzE(entry.Muon_Px[0], entry.Muon_Py[0], entry.Muon_Pz[0], entry.Muon_E[0])
        Muon2 = rt.TLorentzVector()
        Muon2.SetPxPyPzE(entry.Muon_Px[1], entry.Muon_Py[1], entry.Muon_Pz[1], entry.Muon_E[1])
        diMuon = Muon1 + Muon2
        h_masa2.Fill(diMuon.M())


# ========================================
# NORMALIZACIÓN DE HISTOGRAMAS
# ========================================
# Crear leyenda para histogramas de MET
legend = rt.TLegend(0.6,0.9,0.9,0.75) # x1,y1,x2,y2
legend.SetHeader("Legend","C")
legend.AddEntry(h,"Histograma para DY","l")
legend.AddEntry(h2,"Histograma para datos","l")

# Normalizar histogramas de MET (dividir por integral para comparar formas)
h.Scale(1.0/h.Integral())
print("Integral de h2: ", h2.Integral())
h2.Scale(1.0/h2.Integral())

# ========================================
# PAD 1: Comparación de MET (DY vs Datos)
# ========================================
canvas.cd(1)  # Activar primer pad
h.Draw("hist")  # Dibujar DY
h2.Draw("hist same")  # Superponer Datos
legend.Draw()

# ========================================
# PAD 2: Ratio de MET (Datos/DY)
# ========================================
canvas.cd(2)  # Activar segundo pad
h_ratio_met = h2.Clone("h_ratio_met")  # Clonar histograma de datos
h_ratio_met.Divide(h)  # Dividir bin por bin: Datos/DY
h_ratio_met.SetTitle("Ratio MET (Datos/DY)")
h_ratio_met.SetYTitle("Datos/DY")
h_ratio_met.SetLineColor(rt.kBlack)
h_ratio_met.SetMarkerStyle(20)  # Marcadores circulares
h_ratio_met.SetMarkerSize(0.8)
h_ratio_met.Draw("E")  # Dibujar con barras de error
# Línea de referencia en y=1 (indica acuerdo perfecto)
line_met = rt.TLine(0, 1, 80, 1)
line_met.SetLineColor(rt.kRed)
line_met.SetLineStyle(2)  # Línea punteada
line_met.Draw("same")

# ========================================
# PAD 3: Comparación de Masa Invariante (DY vs Datos)
# ========================================
# Crear leyenda para histogramas de masa
legend2 = rt.TLegend(0.7,0.9,0.9,0.75) # x1,y1,x2,y2
legend2.SetHeader("Legend","C")
legend2.AddEntry(h_masa,"Masa invariante para DY","l")
legend2.AddEntry(h_masa2,"Masa invariante para datos","l")

# Normalizar histogramas de masa invariante
h_masa.Scale(1.0/h_masa.Integral())
h_masa2.Scale(1.0/h_masa2.Integral())

canvas.cd(3)  # Activar tercer pad
h_masa.Draw("hist")  # Dibujar DY
h_masa2.Draw("hist same")  # Superponer Datos
legend2.Draw()

# ========================================
# PAD 4: Ratio de Masa Invariante (Datos/DY)
# ========================================
canvas.cd(4)  # Activar cuarto pad
h_ratio_masa = h_masa2.Clone("h_ratio_masa")  # Clonar histograma de datos
h_ratio_masa.Divide(h_masa)  # Dividir bin por bin: Datos/DY
h_ratio_masa.SetTitle("Ratio Masa Invariante (Datos/DY)")
h_ratio_masa.SetYTitle("Datos/DY")
h_ratio_masa.SetLineColor(rt.kBlack)
h_ratio_masa.SetMarkerStyle(20)
h_ratio_masa.SetMarkerSize(0.8)
h_ratio_masa.Draw("E")  # Dibujar con barras de error (propagación estadística)
# Línea de referencia en y=1
line_masa = rt.TLine(0, 1, 140, 1)
line_masa.SetLineColor(rt.kRed)
line_masa.SetLineStyle(2)
line_masa.Draw("same")

# Mostrar el canvas completo
canvas.Draw()